# പ്രാഥമിക അംഗം മിഡിൽവെയർ ഉൾപ്പടെ ഹോട്ടൽ ബുക്കിംഗ്

ഈ നോട്ട് ബുക്ക് മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്കിന്റെ **ഫങ്ഷൻ അടിസ്ഥാനമാക്കിയ മിഡിൽവെയർ**δειച്ചുന്നതാണ്. നാം കൺഡീഷണൽ വർക്ഫ്‌ളോ ഉദാഹരണത്തെ അടിസ്ഥാനമാക്കി പ്രാഥമിക അംഗങ്ങൾക്ക് പ്രത്യേക привилേജുകൾ നൽകുന്ന ഒരു മിഡിൽവെയർ ലെയർ ചേർക്കുന്നു.

## നിങ്ങൾക്ക് പഠിക്കാനാകും:
1. **ഫങ്ഷൻ അടിസ്ഥാനമാക്കിയ മിഡിൽവെയർ**: ഫങ്ഷൻ ഫലങ്ങൾ തടഞ്ഞു മാറ്റുക  
2. **കോൺടെക്സ്റ്റ് ആക്‌സസ്**: പ്രവർത്തനത്തിന് ശേഷം `context.result` വായിക്കുകയും മാറ്റുകയും ചെയ്യുക  
3. **ബിസിനസ് ലജിക് നടപ്പാക്കൽ**: പ്രാഥമിക അംഗങ്ങളുടെ പ്രയോജനം  
4. **ഫലം മറികടക്കൽ**: ഉപയോക്തा നിലയെ അടിസ്ഥാനമാക്കി ഫങ്ഷൻ ഫലം മാറ്റുക  
5. **ഒരേ വർക്ഫ്‌ളോ, വ്യത്യസ്ത ഫലങ്ങൾ**: മിഡിൽവെയർ-ഏതെങ്കിലുമുള്ള പെരുമാറ്റ മാറ്റങ്ങൾ  

## മിഡിൽവെയർ ഉൾപ്പെടെയുള്ള വർക്ഫ്‌ളോ ആര്‍ക്കിടെക്ചർ:

```
User Input: "I want to book a hotel in Paris"
                    ↓
        [availability_agent]
        - Calls hotel_booking tool
        - 🌟 priority_check middleware intercepts
        - Checks user membership status
        - IF priority + no rooms → Override to available!
        - Returns BookingCheckResult
                    ↓
        Conditional Routing
           /                    \
    [has_availability]    [no_availability]
          ↓                      ↓
    [booking_agent]        [alternative_agent]
    (Priority override!)   (Regular users)
          ↓                      ↓
       [display_result executor]
```

## കൺഡീഷണൽ വർക്ഫ്‌ളോയുമായി പ്രധാന വ്യത്യാസം:

**മിഡിൽവെയർ ഇല്ലാതെ** (14-conditional-workflow.ipynb):  
- പാരീസ്-ൽ മുറികൾ ഇല്ല → alternative_agent-നേയ്ക്ക് റൂട്ടുചെയ്യുക  

**മിഡിൽവെയർ ഉൾപ്പെടെ** (ഈ നോട്ട്‌ബുക്ക്):  
- സാധാരണ ഉപയോക്താവ് + പാരീസ് → മുറികൾ ഇല്ല → alternative_agent-നേയ്ക്ക് റൂട്ടുചെയ്യുക  
- പ്രാഥമിക ഉപയോക്താവ് + പാരീസ് → 🌟 മിഡിൽവെയർ മറികടക്കും! → ലഭ്യമാണ് → booking_agent-നേയ്ക്ക് റൂട്ടുചെയ്യുക  

## മുൻകൂർ ആവശ്യകതകൾ:  
- മൈക്രോസോഫ്റ്റ് ഏജന്റ് ഫ്രെയിംവർക്ക് ഇൻസ്റ്റാൾ ചെയ്തിരിക്കുക  
- കൺഡീഷണൽ വർക്ഫ്‌ളോകൾ അറിയുക (കാണുക 14-conditional-workflow.ipynb)  
- GitHub ടോക്കൺ അല്ലെങ്കിൽ OpenAI API കീ  
- മിഡിൽവെയർ പാറ്റേണുകൾക്കുള്ള അടിസ്ഥാന ബോധം


In [1]:
import asyncio
import json
import os
from collections.abc import Awaitable, Callable
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    FunctionInvocationContext,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## Step 1: ഘടനയുള്ള ഔട്ട്പുട്ടുകൾക്കായി പൈഡാൻടിക് മോഡലുകൾ നിർവ്വചിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരിച്ചറിയേണ്ട **സ്കീമ** നിർവ്വചിക്കുന്നു. മിഡിൽവെയർ ലഭ്യത ഫലത്തിൽ മാറ്റം വരുത്തുന്നപ്പോൾ ട്രാക്ക് ചെയ്യാൻ ഞങ്ങൾ `priority_override` ഫീൽഡ് ചേർത്തിട്ടുണ്ട്.


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str
    priority_override: bool = False  # 🆕 NEW! Tracks if middleware overrode the result


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check with priority_override)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check with priority_override)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## Step 2: പ്രാധാന്യമുള്ള അംഗങ്ങളുടെ ഡേറ്റാബേസ് നിർവചിക്കുക

ഈ ഡെമോക്കായി, നാം ഒരു പ്രാധാന്യമുള്ള അംഗരാജ്യ ഡേറ്റാബേസ് അനുകരിക്കും. പ്രൊഡക്ഷനിൽ, ഇത് ഒരു യഥാർത്ഥ ഡേറ്റാബേസ് അല്ലെങ്കിൽ API-നെ അഭ്യർത്ഥിക്കും.

**പ്രാധാന്യമുള്ള അംഗങ്ങൾ:**
- `alice@example.com` - VIP അംഗം
- `bob@example.com` - പ്രീമിയം അംഗം  
- `priority_user` - ടെസ്റ്റ് അക്കൗണ്ട്


In [3]:
# Simulated priority members database
PRIORITY_MEMBERS = {
    "alice@example.com",
    "bob@example.com",
    "priority_user",
}

# Global variable to track current user (in real app, use proper session management)
current_user_id = "regular_user"  # Default: regular user


def set_user(user_id: str):
    """Set the current user for the session."""
    global current_user_id
    current_user_id = user_id
    is_priority = user_id in PRIORITY_MEMBERS
    status = "🌟 PRIORITY MEMBER" if is_priority else "👤 Regular User"

    display(
        HTML(f"""
        <div style='padding: 15px; background: {"linear-gradient(135deg, #FFD700 0%, #FFA500 100%)" if is_priority else "#e3f2fd"}; 
                    border-left: 4px solid {"#FF6B35" if is_priority else "#2196f3"}; border-radius: 4px; margin: 10px 0;'>
            <strong>👤 Current User Set:</strong> {user_id}<br>
            <strong>Status:</strong> {status}
        </div>
    """)
    )


print("✅ Priority members database created")
print(f"   Priority members: {len(PRIORITY_MEMBERS)} users")

✅ Priority members database created
   Priority members: 3 users


## ചുവടു 3: ഹോട്ടൽ ബുക്കിംഗ് ടൂൾ സൃഷ്‌ടിക്കുക

ക്രമീകരണ പ്രവാഹം പോലെ തന്നെ, പക്ഷേ ഇപ്പോൾ ഇത് മിഡിൽവെയർ തടഞ്ഞുറപ്പിക്കും!


In [4]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Step 4: 🌟 മുൻഗണനാ പരിശോധന മിഡിൽവെയർ സൃഷ്ടിക്കുക (അവസാന അടിസ്ഥാനം!)

ഈ നോട്ട്‌ബുക്കിന്റെ **പ്രധാന പ്രവർത്തനം** ഇതാണ്. മിഡിൽവെയർ:

1. ഹോട്ടൽ_ബുക്കിംഗ് ഫังก്ഷൻ കോൾ **അടയാളപ്പെടുത്തുന്നു**
2. സാധാരണമായി `next(context)` കോൾ ചെയ്ത് ഫംഗ്ഷൻ **നടത്തുന്നു**
3. `context.result` ൽ ഫലം **പരിശോധിക്കുന്നു**
4. ഉപയോക്താവ് മുൻഗണനയുള്ള പക്ഷം മുറികൾ ലഭ്യമല്ലെങ്കിൽ ഫലം **മാറുന്നു**
5. മാറ്റിയ ഫലം ഏജന്റിന് **പിന്നീട लौटിക്കുന്നു**

**പ്രധാന മാതൃക:**
```python
async def my_middleware(context, next):
    await next(context)  # ഫങ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    # ഇപ്പോൾ context.result ഫങ്ഷന്റെ ഔട്ട്പുട്ട് അടങ്ങിയിരിക്കുന്നു
    if some_condition:
        context.result = new_value  # മറികടക്കുക!
```


In [5]:
async def priority_check_middleware(
    context: FunctionInvocationContext,
    next: Callable[[FunctionInvocationContext], Awaitable[None]],
) -> None:
    """
    Function middleware that overrides hotel_booking results for priority members.
    
    Workflow:
    1. Let the function execute normally
    2. Check if user is a priority member
    3. If priority + no availability → Override to make rooms available!
    4. Agent will then route to booking path instead of alternative path
    """
    function_name = context.function.name

    display(
        HTML(f"""
        <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Middleware:</strong> Intercepting {function_name}...
        </div>
    """)
    )

    # Execute the original function
    await next(context)

    # Now inspect and potentially modify the result
    if context.result and function_name == "hotel_booking":
        result_data = json.loads(context.result)
        destination = result_data.get("destination", "")
        has_availability = result_data.get("has_availability", False)

        # Check if user is priority member
        is_priority = current_user_id in PRIORITY_MEMBERS

        # Override logic: Priority member + no availability → Make available!
        if is_priority and not has_availability:
            display(
                HTML(f"""
                <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); 
                            border-radius: 8px; margin: 10px 0; box-shadow: 0 4px 12px rgba(255,165,0,0.4);'>
                    <h3 style='margin: 0 0 10px 0; color: #333;'>🌟 PRIORITY OVERRIDE ACTIVATED! 🌟</h3>
                    <p style='margin: 0; color: #555; line-height: 1.6;'>
                        <strong>User:</strong> {current_user_id}<br>
                        <strong>Status:</strong> VIP Priority Member<br>
                        <strong>Action:</strong> Overriding "No Availability" for {destination}<br>
                        <strong>Result:</strong> ✅ Rooms now available for priority booking!
                    </p>
                </div>
            """)
            )

            # Override the result!
            result_data["has_availability"] = True
            result_data["priority_override"] = True
            context.result = json.dumps(result_data)

        elif not has_availability:
            display(
                HTML(f"""
                <div style='padding: 12px; background: #ffebee; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>ℹ️ Middleware:</strong> No priority override (user: {current_user_id})
                </div>
            """)
            )


print("✅ priority_check_middleware created")
print("   - Intercepts hotel_booking function")
print("   - Overrides availability for priority members")

✅ priority_check_middleware created
   - Intercepts hotel_booking function
   - Overrides availability for priority members


## പടി 5: റൂട്ടിംഗിനുള്ള നിബന്ധന ഫംഗ്ഷനുകൾ നിർവചിക്കുക

നിബന്ധന സാമ്യമുള്ള പ്രവാഹത്തിലുള്ള ഫംഗ്ഷനുകളുപോലെ - അവർ ഘടനയിട്ട ഔട്ട്പുട്ട് നിരീക്ഷിച്ച് റൂട്ടിംഗ് നിർണയിക്കുന്നു.


In [6]:
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available (including priority overrides!)."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        # Check if this was a priority override
        override_indicator = " 🌟" if result.priority_override else ""

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}{override_indicator}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception:
        return False


print("✅ Condition functions defined")

✅ Condition functions defined


## Step 6: കസ്റ്റം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്‌ടിക്കുക

മുമ്പത്തെപ്പോലെ തന്നെ എക്സിക്യൂട്ടർ - അന്തിമ വർക്ക്‌ഫ്ലോ ഔട്ട്പുട്ട് പ്രദർശിപ്പിക്കുന്നു.


In [7]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """Display the final result as workflow output."""
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created")

✅ display_result executor created


## Step 7: വാതावरण ചുരുങ്ങിയ വേരിയബിളുകൾ ലോഡുചെയ്യുക

LLM ക്ലയന്റിനെ കോൺഫിഗർ ചെയ്യുക (GitHub മോഡലുകൾ അല്ലെങ്കിൽ OpenAI).


In [8]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")


## അടിയന്തര 8: മിഡിൽവെയർ ഉപയോഗിച്ച് AI ഏജന്റുമാർ സൃഷ്ടിക്കുക

**പ്രധാന വ്യത്യാസം:** availability_agent സൃഷ്ടിക്കുമ്പോൾ, `middleware` പാരാമീറ്റർ നാം കൈമാറുകയാണ്!

വിവിധ പ്രവർത്തന ഘടകങ്ങളിലേക്ക് priority_check_middleware നമുക്ക് ഏജന്റിന്റെ ഫംഗ്ഷൻ വിളിപ്പിക്കുന്ന പൈപ്പ്‌ലൈൻ വൺഗുന്ന വിധം ഇന്ജെക്ട് ചെയ്യുന്നത് ഇങ്ങനെ ആണ്.


In [9]:
# Agent 1: Check availability with tool + middleware
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), message (string), "
            "and priority_override (bool, true if priority member got special access). "
            "The message should summarize the availability status and mention if priority override occurred."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
        middleware=[priority_check_middleware],  # 🌟 MIDDLEWARE INJECTION!
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "If priority_override is true in the input, mention that they received priority member access. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - WITH priority_check_middleware 🌟</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 9: പ്രവൃത്തി പ്രവാഹം നിർമ്മിക്കുക

മുൻപത്തെ പോലെ തന്നെ പ്രവൃത്തി പ്രവാഹ ഘടന - ലഭ്യതയെ അടിസ്ഥാനമാക്കി നിബന്ധനാപൂര്‍വ്വമായ റൂട്ടിംഗ്.


In [10]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH (can be triggered by middleware override!)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing (Middleware-Aware):</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> (or 🌟 <strong>priority override</strong>) → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 10: ടെസ്റ്റ് കേസ് 1 - പാരിസിലെ സാധാരണ ഉപയോക്താവ് (ഓവർറൈഡ് ഇല്ല)

ഒരു സാധാരണ ഉപയോക്താവ് പാരിസ് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → മുറികളില്ല → alternative_agent ലേക്ക് റൂട്ടുചെയ്യുന്നു


In [11]:
# Set as regular user
set_user("regular_user")

display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Regular User + Paris</h3>
        <p style='margin: 0;'><strong>Expected:</strong> No rooms → No middleware override → Alternative suggestion</p>
    </div>
""")
)

# Create request
request_regular = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_regular = await workflow.run(request_regular)
outputs_regular = events_regular.get_outputs()

# Display results
if outputs_regular:
    result_regular = AlternativeResult.model_validate_json(outputs_regular[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: #fff; border: 2px solid #ff9800; border-radius: 12px; margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #e65100;'>📊 RESULT (Regular User)</h3>
            <div style='background: #fff3e0; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0;'><strong>Middleware:</strong> No priority override (regular user)</p>
                <p style='margin: 0 0 10px 0;'><strong>Alternative:</strong> 🏨 {result_regular.alternative_destination}</p>
                <p style='margin: 0;'><strong>Reason:</strong> {result_regular.reason}</p>
            </div>
        </div>
    """)
    )

## Step 11: Test Case 2 - 🌟 പ്രാഥമിക ഉപയോക്താവ് പാരിസിൽ (ഓവർറൈഡ് ഉപയോഗിച്ച്!)

ഒരു പ്രാധാന്യമുള്ള അംഗം പാരിസിലേക്ക് ബുക്ക് ചെയ്യാൻ ശ്രമിക്കുന്നു → ആദ്യം റൂമുകൾ available ഇല്ല → 🌟 മിഡിൽവെയർ ഓവർറൈഡ് ചെയ്യുന്നു! → ബുക്കിംഗ് ഏജന്റിലേക്ക് റൂട്ടുചെയ്യുന്നു

**ഇതാണു മിഡിൽവെയറിന്റെ ശക്തിയുടെ പ്രധാന പ്രദർശനം!**


In [12]:
# Set as priority user
set_user("priority_user")

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #333;'>🧪 TEST CASE 2: 🌟 Priority User + Paris</h3>
        <p style='margin: 0; color: #555;'><strong>Expected:</strong> No rooms → 🌟 MIDDLEWARE OVERRIDE → Rooms available → Booking suggestion!</p>
    </div>
""")
)

# Create request
request_priority = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run workflow
events_priority = await workflow.run(request_priority)
outputs_priority = events_priority.get_outputs()

# Display results
if outputs_priority:
    result_priority = BookingConfirmation.model_validate_json(outputs_priority[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; 
                    box-shadow: 0 8px 16px rgba(255,165,0,0.4); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 RESULT (Priority Member) 🌟</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Priority Override!)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> 🌟 OVERRIDE ACTIVATED!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_priority.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_priority.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_priority.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #fff3cd; border-radius: 6px; border-left: 4px solid #FF6B35;'>
                    <strong>💡 What Just Happened:</strong><br>
                    1. hotel_booking tool returned "no availability"<br>
                    2. priority_check_middleware intercepted the result<br>
                    3. Middleware checked user status: priority_user ✅<br>
                    4. Middleware OVERRODE the result to "available"<br>
                    5. Workflow routed to booking_agent instead of alternative_agent!
                </div>
            </div>
        </div>
    """)
    )

## ഘട്ടം 12: ടെസ്റ്റ് കേസ് 3 - സ്റ്റോക്ക്ഹോംിലുള്ള പ്രാധാന്യ ഉപയോക്താവ് (ആഗോളമായി ലഭ്യമാണ്)

പ്രാധാന്യ ഉപയോക്താവ് സ്റ്റോക്ക്ഹോം ശ്രമിക്കുന്നു → മുറികൾ ലഭ്യമാണ് → ഓവർറൈഡ് ആവശ്യമില്ല → booking_agent ൽ റൂട്ടുചെയ്യുന്നു

ഇത് മിഡിൽവെയർ ആവശ്യപ്പെട്ടപ്പോൾ മാത്രമേ പ്രവർത്തിക്കുകയുള്ളൂ എന്ന് കാണിക്കുന്നു!


In [13]:
# Priority user is still set from previous test

display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 3: Priority User + Stockholm</h3>
        <p style='margin: 0;'><strong>Expected:</strong> Rooms available → No override needed → Booking suggestion</p>
    </div>
""")
)

# Create request
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 RESULT (Priority User - No Override Needed)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available (Natural)</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Middleware:</strong> No override needed</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <div style='margin-top: 15px; padding: 15px; background: #e8f5e9; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <strong>💡 Middleware Behavior:</strong><br>
                    • hotel_booking returned "available" naturally<br>
                    • Middleware saw available = true → No override needed<br>
                    • Workflow proceeded normally to booking_agent
                </div>
            </div>
        </div>
    """)
    )

## പ്രധാനമായ സാരാംശങ്ങളും മിഡിൽവെയർ ആശയങ്ങളുമൊക്കെയാണ്

### ✅ നിങ്ങൾ പഠിച്ചതെന്തെല്ലാം:

#### **1. ഫങ്ഷൻ അധിഷ്ഠിത മിഡിൽവെയർ പാറ്റേൺ**

മിഡിൽവെയർ ലളിതമായ അസിങ്ക്രൺ ഫങ്ഷൻ ഉപയോഗിച്ച് ഫങ്ഷൻ കോൾസ് തടയുന്നു:

```python
async def my_middleware(
    context: FunctionInvocationContext,
    next: Callable,
) -> None:
    # ഫംഗ്ഷൻ പ്രവർത്തനം ആരംഭിക്കുന്നതിന് മുമ്പ്
    print("Intercepting...")
    
    # ഫംഗ്ഷൻ പ്രവർത്തിപ്പിക്കുക
    await next(context)
    
    # ഫംഗ്ഷൻ പ്രവർത്തനം കഴിഞ്ഞതിന് ശേഷം - ഫലം പരിശോധിക്കുക
    if context.result:
        # ആവശ്യമെങ്കിൽ ഫലം മാറ്റിവെക്കുക
        context.result = modified_value
```

#### **2. കോൺടെക്സ്റ്റ് ആക്സസ് ചെയ്യൽ, ഫലം മറിച്ചെഴുതൽ**

- `context.function` - കോൾ ചെയ്യപ്പെടുന്ന ഫങ്ഷൻ ആക്സസ് ചെയ്യുക  
- `context.arguments` - ഫങ്ഷൻ ആർഗ്യുമെന്റുകൾ വായിക്കുക  
- `context.kwargs` - അധിക പാരാമീറ്ററുകൾ ആക്സസ് ചെയ്യുക  
- `await next(context)` - ഫങ്ഷൻ പ്രവർത്തിപ്പിക്കുക  
- `context.result` - ഫങ്ഷന്റ്റെ ഔട്ട്‌പുട്ട് വായിക്കാനോ/മാറ്റാനോ  

#### **3. ബിസിനസ് ലോജിക് നടപ്പാക്കൽ**

ഞങ്ങളുടെ മിഡിൽവെയർ പ്രയറിറ്റി മെമ്പർ ബനിഫിറ്റുകൾ നൽകുന്നു:  
- **റഗുലർ യൂസേഴ്സ്**: മാറ്റമില്ലാതെ സാധാരണ പ്രവൃത്തി  
- **പ്രയറിറ്റി യൂസേഴ്സ്**: "ലഭ്യതയില്ല" → "ലഭ്യമാണ്" എന്ന് മറിച്ചെഴുതൽ  
- **നിബന്ധനാപരമായ ലോജിക്**: ആവശ്യമുള്ളപ്പോൾ മാത്രമാകും മാറ്റം വരുത്തുന്നത്  

#### **4. ഒരേ പ്രവൃത്തി പ്രവൃത്തി, വ്യത്യസ്ത ഫലം**

മിഡിൽവെയർ ശക്തി:  
- ✅ വർക്ക്‌ഫ്ലോ ഘടനയിൽ മാറ്റമില്ല  
- ✅ ടൂൾ ഫങ്ഷനിൽ മാറ്റമില്ല  
- ✅ നിബന്ധനാപരമായ റൂട്ടിങ് ലോജിക് മാറ്റമില്ല  
- ✅ മിഡിൽവെയർ മാത്രം → വ്യത്യസ്ത പെരുമാറ്റം!

### 🚀 യാഥാർത്ഥ്യപ്രയോഗങ്ങൾ:

1. **വിഐപി/പ്രീമിയം ഫീച്ചറുകൾ**  
   - പ്രീമിയം യൂസർമാർക്ക് റേറ്റ് ലിമിറ്റുകൾ മറിച്ചെഴുതുക  
   - റിസോഴ്സുകൾക്ക് പ്രायरിറ്റി ആക്‌സസ് നൽകുക  
   - പ്രീമിയം സവിശേഷതകൾ ഡൈനാമിക്കായി അൺലോക്ക് ചെയ്യുക  

2. **എ/ബി ടെസ്റ്റിങ്**  
   - വ്യത്യസ്ത പ്രവർത്തനങ്ങളിൽ യൂസർമാർ റീറ്റു ചെയ്യുക  
   - ചില യൂസർമാരോടൊപ്പം പുതിയ ഫീച്ചറുകൾ പരീക്ഷിക്കുക  
   - постепенная ഫീച്ചര്‍ റോളൗട്ട്  

3. **സുരക്ഷയും അനുസരണയും**  
   - ഫങ്ഷൻ കോൾസ് ഓഡിറ്റ് ചെയ്യുക  
   - സെൻസിറ്റീവ് പ്രവർത്തനങ്ങൾ തടയുക  
   - ബിസിനസ് നിയമങ്ങൾ നടപ്പിലാക്കുക  

4. **പ്രകടനം മെച്ചപ്പെടുത്തൽ**  
   - ചില യൂസർമാർക്കായി ഫലം കാഷ് ചെയ്യുക  
   - സാധ്യതയുള്ളപ്പോൾ വിലയേറിയ പ്രവർത്തനങ്ങൾ ഒഴിവാക്കുക  
   - ഡൈനാമിക് റിസോഴ്സ് ആയോജനം  

5. **പിശകുകളെ കൈകാര്യംചെയ്യൽ & റിട്രൈ**  
   - പിശകുകൾ പിടികൂടി ശാന്തതയോടെ കൈകാര്യം ചെയ്യുക  
   - റിട്രൈ ലോജിക് നടപ്പിലാക്കുക  
   - പകരം ഉപയോഗിക്കാനുള്ള മറ്റു നടപടികൾക്ക് fallback ചെയ്യുക  

6. **ലോഗിങ്ങും നിരീക്ഷണവും**  
   - ഫങ്ഷൻ പ്രവർത്തന സമയങ്ങൾ ട്രാക്ക് ചെയ്യുക  
   - പാരാമീറ്ററുകളും ഫലങ്ങളും ലോഗ് ചെയ്യുക  
   - ഉപയോഗ മാതൃകകൾ നിരീക്ഷിക്കുക  

### 🔑 ഡെക്കറേറ്ററുകളിൽ നിന്ന് പ്രധാന വ്യത്യാസങ്ങൾ:

| സവിശേഷത | ഡെക്കറേറ്റർ | മിഡിൽവെയർ |
|---------|-----------|------------|
| **ആവരണം** | ഒറ്റ ഫങ്ഷൻ | എജന്റിലെ എല്ലാ ഫങ്ഷനുകളും |
| **സ്ഥിതിസ്വാതന്ത്ര്യം** | നിർവാചന സമയത്ത് നിശ്ചിതം | റൺടൈമിൽ ഡൈനാമിക് |
| **കോൺടെക്സ്റ്റ്** | പരിമിതം | പൂർണ എജന്റ് കോൺടെക്സ്റ്റ് |
| **സംയോജനം** | ഒട്ടനവധി ഡെക്കറേറ്ററുകൾ | മിഡിൽവെയർ പൈപ്പ്‌ലൈൻ |
| **എജന്റ് അവഗാഹി** | ഇല്ല | ഉണ്ട് (എജന്റ് സ്റ്റേറ്റ് ആക്സസ്) |

### 📚 എപ്പോൾ മിഡിൽവെയർ ഉപയോഗിക്കണം:

✅ **മിഡിൽവെയർ ഉപയോഗിക്കുക:**
- യൂസർ/സെഷൻ സ്റ്റേറ്റിനായി പെരുമാറ്റം മാറ്റേണ്ടിവരുമെങ്കിൽ  
- ഒന്നിൽ അധികം ഫങ്ഷനുകളിൽ ലോജിക് പ്രയോഗിക്കാനുണ്ടെങ്കിൽ  
- എജന്റ് നിലയിൽ കോൺടെക്സ്റ്റ് ആക്‌സസ് വേണമെങ്കിൽ  
- ക്രോസ്-കട്ടിംഗ് ആശങ്കകൾ (ലോഗിങ്ങ്, ഓത്ത്, മുതലായവ) നടപ്പിലാക്കുമ്പോൾ  

❌ **മിഡിൽവെയർ ഉപയോഗിക്കേണ്ടാ:**
- ലളിതമായ ഇൻപുട്ട് പരിശോധന (പൈഡാൻറിക്ക് ഉപയോഗിക്കുക)  
- ഫങ്ഷൻ-നിർദിഷ്ട ലോജിക് (ഫങ്ഷനിൽ തന്നെ വയ്ക്കുക)  
- ഒറ്റ തവണ മാറ്റങ്ങൾ (സൂചിപ്പിച്ച ഫങ്ഷൻ മാറ്റുക)  

### 🎓 ഉയർന്ന നിലവാരത്തിലുള്ള പാറ്റേണുകൾ:

```python
# ഒന്നിലധികം മിഡിൽവെയർ (പ്രവർത്തനക്രമം പ്രധാനമാണ്!)
middleware=[
    logging_middleware,      # ആദ്യമായി ലോഗുകൾ
    auth_middleware,         # പിന്നെ ആധികാരം പരിശോധിക്കുന്നു
    cache_middleware,        # പിന്നെ കാഷെ പരിശോധിക്കുന്നു
    rate_limit_middleware,   # പിന്നെ നിരക്ക് പരിധികൾ സജ്ജമാക്കുന്നു
    priority_check_middleware  # അവസാനത്തിലും പ്രാധാന്യം പരിശോധിക്കുന്നു
]

# നിബന്ധനാത്മക മിഡിൽവെയർ പ്രവർത്തനം
async def conditional_middleware(context, next):
    if should_execute(context):
        await next(context)
        # ഫലം മാറ്റം വരുത്തുക
    else:
        # മുഴുവൻ പ്രവർത്തനം ഒഴിക്കൂ
        context.result = cached_value
```

### 🔗 ബന്ധപ്പെട്ട ആശയങ്ങൾ:

- **എജന്റ് മിഡിൽവെയർ**: agent.run() കോൾസ് തടയുന്നു  
- **ഫങ്ഷൻ മിഡിൽവെയർ**: ടൂൾ ഫങ്ഷൻ കോൾസ് തടയുന്നു (ഞങ്ങൾ ഉപയോഗിച്ചത്!)  
- **മിഡിൽവെയർ പൈപ്പ്‌ലൈൻ**: മിഡിൽവെയറുകളുടെ ശ്രേണി  
- **കോൺടെക്സ്റ്റ് പ്രോപ്പഗേഷൻ**: സ്റ്റേറ്റ് മിഡിൽവെയർ ശൃംഖലയിലൂടെ കൈമാറുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**വിവരണക്കുറിപ്പ്**:
ഈ രേഖ AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്യപ്പെട്ടതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി പരിശ്രമിച്ചിട്ടുമുണ്ടെങ്കിലും, യന്ത്ര വിവർത്തനത്തിൽ പിഴവുകൾ അല്ലെങ്കിൽaccuracyത്വത്വക്കുറവുകൾ ഉണ്ടാകാവുന്നതാണ്. സ്വNative ഭാഷയിലെ ആദ്യ രേഖ ഔദ്യോഗിക ഉറവിടമായിരിക്കുകയാണ്. നിർണായക വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം സ്വീകരിക്കേണ്ടതാണ്. ഈ വിവർത്തനം ഉപയോഗിക്കുന്നതിലൂടെ സംഭവിക്കുന്ന തെറ്റായ അറിവുകൾക്കോ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കോ ഞങ്ങൾ ഉത്തരവാദിത്തം ഏറ്റെടുക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
